## 1. Fast Evaluation Patch
Run this cell to evaluate your saved `.pt` models without retraining! Make sure you run the dataset and model definition cells in your main notebook first.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import time
from tqdm import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.ops import box_iou

# --- CONFIGURATION ---
MODEL_TYPE = 'mobilenet'  # Change to 'resnet18' to test ResNet
PT_FILE = 'mobilenet_ssd_hagrid_detector.pt' # Change to 'resnet18_hagrid_detector.pt' for ResNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if MODEL_TYPE == 'mobilenet':
    eval_model = MobileNetLocalization(num_classes=5).to(device)
else:
    eval_model = ResNetLocalization(num_classes=5).to(device)

print(f"Loading {PT_FILE}...")
checkpoint = torch.load(PT_FILE, map_location=device)
eval_model.load_state_dict(checkpoint['model_state_dict'])
eval_model.eval()
print("Model loaded successfully!")

print("\nEvaluating on Test Set (this won't OOM!)...")
metric = MeanAveragePrecision(iou_type="bbox").to(device)
ious, all_lat_ms = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, leave=False):
        x = batch["image"].to(device)
        target_boxes = batch["target_box"].to(device)
        target_labels = batch["target_label"].to(device)
        
        if device.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        box_preds, cls_logits = eval_model(x)
        if device.type == "cuda": torch.cuda.synchronize()
        t1 = time.perf_counter()
        all_lat_ms.extend([(t1 - t0) * 1000.0 / x.size(0)] * x.size(0))
        
        p_l, p_s, p_b = decode_batch_predictions(box_preds, cls_logits)
        
        g_xc, g_yc, g_w, g_h = target_boxes.unbind(1)
        g_b = torch.stack([g_xc - g_w/2, g_yc - g_h/2, g_xc + g_w/2, g_yc + g_h/2], dim=1).clamp(0, 1)
        g_l = target_labels.cpu().tolist()
        
        ious.extend(torch.diag(box_iou(p_b.cpu(), g_b.cpu())).numpy().tolist())
        
        preds, target = [], []
        for j in range(len(p_l)):
            preds.append({"boxes": p_b[j].unsqueeze(0).to(device) * 384.0, "scores": torch.tensor([p_s[j]], device=device), "labels": torch.tensor([p_l[j]], device=device)})
            target.append({"boxes": g_b[j].unsqueeze(0).to(device) * 384.0, "labels": torch.tensor([g_l[j]], device=device)})
        metric.update(preds, target)

mAP_dict = metric.compute()
print("\n" + "="*40)
print(f"[{MODEL_TYPE.upper()}] FINAL TEST RESULTS")
print("="*40)
print(f"mAP50:        {mAP_dict['map_50'].item():.4f}")
print(f"mean_IoU:     {float(np.mean(ious)):.4f}")
print(f"Mean Latency: {float(np.mean(all_lat_ms)):.2f} ms per image")
print("="*40)

print("\n--- Formal Mathematical Output Preview ---")
with torch.no_grad():
    batch = next(iter(test_loader))
    x = batch["image"].to(device)
    box_preds, cls_logits = eval_model(x)
    p_l, p_s, p_b = decode_batch_predictions(box_preds, cls_logits)
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for i in range(4):
        img_np = x[i].cpu().permute(1, 2, 0).numpy()
        score = p_s[i]
        label = p_l[i]
        x1, y1, x2, y2 = p_b[i].cpu().tolist()
        w_b, h_b = max(0.0, x2 - x1), max(0.0, y2 - y1)
        
        gesture = target_classes[label]
        # Mathematical yi = (o, [x, y, w, h], c)
        print(f"Image {i+1}: y_i = ({score:.3f}, [{x1:.3f}, {y1:.3f}, {w_b:.3f}, {h_b:.3f}], {label + 1})  # {gesture}")
        
        ax = axes[i]
        ax.imshow(img_np)
        ax.add_patch(plt.Rectangle((x1*384, y1*384), w_b*384, h_b*384, fill=False, color='lime', linewidth=2))
        ax.set_title(f"{gesture} (Conf: {score:.2f})")
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(f"{MODEL_TYPE}_preview.png")
    plt.show()
    print(f"\nSaved preview image to {MODEL_TYPE}_preview.png")
